# Classification modelling

## Libraries

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.data_cleaning_and_manipulations import fix_data_types, handle_missing_values, handle_outliers, build_probability_table
from src.data_cleaning_and_manipulations import drop_unnecessary_columns, run_feature_selection_methods, manipulate_df_process, apply_early_probability
from src.data_cleaning_and_manipulations import compare_xgb_feature_sets
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

## Functions

In [2]:
def train_evaluate_classifier(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    resampling=False,
    random_state=42
):
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.preprocessing import LabelEncoder, label_binarize
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        roc_curve,
        auc,
        roc_auc_score
    )

    # Encode labels
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)

    X_fit = X_train.copy()
    y_fit = y_train_enc.copy()

    # Optional resampling
    if resampling:
        from imblearn.over_sampling import RandomOverSampler

        ros = RandomOverSampler(random_state=random_state)
        X_fit, y_fit = ros.fit_resample(X_fit, y_fit)

        print("Resampling applied.")
        print("Class distribution after resampling:")
        print(pd.Series(y_fit).value_counts().sort_index())
        print("-" * 50)

    # Train model
    model.fit(X_fit, y_fit)

    # Predict
    y_pred = model.predict(X_val)

    print("Classification Report:")
    print(classification_report(
        y_val_enc,
        y_pred,
        target_names=le.classes_
    ))

    # Confusion matrix
    cm = confusion_matrix(y_val_enc, y_pred)

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()

    # ROC curve
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_val)

        y_val_bin = label_binarize(
            y_val_enc,
            classes=np.arange(len(le.classes_))
        )

        plt.figure(figsize=(8, 6))

        for i, class_name in enumerate(le.classes_):
            fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)

            plt.plot(
                fpr,
                tpr,
                label=f"{class_name} (AUC = {roc_auc:.2f})"
            )

        plt.plot([0, 1], [0, 1], "k--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve - One-vs-Rest")
        plt.legend()
        plt.show()

        macro_auc = roc_auc_score(
            y_val_bin,
            y_proba,
            average="macro",
            multi_class="ovr"
        )

        print(f"Macro ROC AUC: {macro_auc:.4f}")

    else:
        print("ROC curve skipped: model does not support predict_proba().")

    return model, le



## Imports

In [3]:
train_df = pd.read_csv(r'../data/model_datasets/train_df.csv', encoding='utf-8-sig')
val_df = pd.read_csv(r'../data/model_datasets/val_df.csv', encoding='utf-8-sig')

## Standardize DFs

In [4]:
train_df = manipulate_df_process(train_df)
train_df = drop_unnecessary_columns(train_df)
val_df = manipulate_df_process(val_df)
val_df = drop_unnecessary_columns(val_df)

## Process

### Add category variable (3 categories)

In [ ]:
train_df = categorize_delay(train_df, col="duration_difference_min")
val_df = categorize_delay(val_df, col="duration_difference_min")
train_df.delay_category.value_counts()

### Preapre model DFs (X,y)

In [ ]:
## Define target variable and X_train and y_train
target_col = "delay_category"
X_train = train_df.drop(columns=[target_col,'duration_difference_min'])
y_train = train_df[target_col]
X_val = val_df.drop(columns=[target_col,'duration_difference_min'])
y_val = val_df[target_col]

### Train models

In [ ]:
## 1. xgBoost

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)


## 2. RandomForest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

### Run models

In [ ]:
### XGBoost & Resampling

model, le = train_evaluate_classifier(
    xgb,
    X_train,
    y_train,
    X_val,
    y_val,
    resampling=True
)

In [ ]:
### XGBoost with resampling

model, le = train_evaluate_classifier(
    xgb,
    X_train,
    y_train,
    X_val,
    y_val,
    resampling=False
)

In [ ]:
## 3. Random forest

model, le = train_evaluate_classifier(
    rf,
    X_train,
    y_train,
    X_val,
    y_val,
    resampling=False
)